# 05 — Preparação para Visualização

**Objetivo:** moldar os dados em formatos prontos para cada tipo de gráfico do dashboard.
**Entrada:** DataFrame de `gerar_metricas()` — formato long com as três métricas calculadas.
**Saída:** dicionário com quatro DataFrames: `heatmap`, `ranking`, `series`, `por_tipo`.

> **Status:** `preparar_visualizacao()` foi removida do `pipeline.py`.
> O dashboard (`app.py`) evoluiu para aplicar os filtros do usuário
> (AISP, período, tipo de crime) diretamente sobre os dados em tempo real,
> o que tornou o pré-processamento estático inviável. As transformações
> exploradas neste notebook — `pivot_table` para heatmap, `nlargest` para
> ranking — continuam documentando os padrões que o dashboard usa
> internamente ao construir cada gráfico.

## Célula 1 — Setup

Rodamos o pipeline completo até `gerar_metricas()` para ter o dado certo.
`analise_temporal` também é importada pois `series` reutiliza sua lógica.

In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ))

from pipeline import carregar_dados, limpar_dados, gerar_metricas, analise_temporal

ARQUIVO = RAIZ / "data" / "raw" / "BaseDPEvolucaoMensalCisp.csv"

df = gerar_metricas(limpar_dados(carregar_dados(ARQUIVO)))
print("Entrada:", df.shape)
df.head(3)

2026-05-11 22:47:19 [INFO] Dados carregados: 37588 linhas, 65 colunas
2026-05-11 22:47:19 [INFO] Limpeza concluida. Shape final: (150352, 8)
2026-05-11 22:47:20 [INFO] Métricas geradas. Shape final: (150352, 12)


Entrada: (150352, 12)


,cisp,mes_ano,aisp,risp,munic,regiao,tipo_crime,qtd_ocorrencias,populacao,taxa_100k,media_movel_3,variacao_pct
75179,6,2003-01,1,1,RIO DE JANEIRO,CAPITAL,cvli,6,232000.0,2.586207,6.000000,NaN
75180,7,2003-01,1,1,RIO DE JANEIRO,CAPITAL,cvli,4,232000.0,1.724138,5.000000,-33.333333
75305,6,2003-02,1,1,RIO DE JANEIRO,CAPITAL,cvli,6,232000.0,2.586207,5.333333,50.000000


## Célula 2 — Heatmap

O heatmap precisa de uma tabela onde linhas são AISPs e colunas são meses.
Cada célula contém a `taxa_100k` — quanto maior, mais escura a cor no gráfico.

`pivot_table()` é a transformação que faz isso: pega o formato long e "abre"
uma coluna em várias colunas, uma por valor único.

| parâmetro | valor | por quê |
|---|---|---|
| `index` | `"aisp"` | cada linha é uma AISP |
| `columns` | `"mes_ano"` | cada coluna é um mês |
| `values` | `"taxa_100k"` | o valor que preenche a célula |
| `aggfunc` | `"sum"` | soma caso haja múltiplos crimes por AISP+mês |
| `fill_value` | `0` | meses sem registro viram 0, não NaN |

In [2]:
heatmap = df.pivot_table(
    index="aisp",
    columns="mes_ano",
    values="taxa_100k",
    aggfunc="sum",
    fill_value=0,
)

print("Shape do heatmap (AISPs x meses):", heatmap.shape)
heatmap.iloc[:5, :5]

Shape do heatmap (AISPs x meses): (42, 279)


mes_ano,2003-01,2003-02,2003-03,2003-04,2003-05
aisp,,,,,
1,12.931034,11.206897,8.189655,16.810345,8.620690
10,16.923077,16.923077,7.692308,16.923077,7.179487
11,0.000000,0.000000,0.000000,0.000000,0.000000
12,0.000000,0.000000,0.000000,0.000000,0.000000
13,0.000000,0.000000,0.000000,0.000000,0.000000


## Célula 3 — Ranking

O ranking responde: *quais são as 10 AISPs mais críticas agora?*

"Agora" = o mês mais recente com dados. Filtramos apenas esse mês,
somamos todas as ocorrências por AISP e pegamos as 10 maiores.

`nlargest(10)` é o equivalente de `sort_values().head(10)`, mas mais direto.

In [3]:
mes_mais_recente = df["mes_ano"].max()
print("Mês de referência:", mes_mais_recente)

ranking = (
    df[df["mes_ano"] == mes_mais_recente]
    .groupby("aisp")["qtd_ocorrencias"]
    .sum()
    .nlargest(10)
    .reset_index()
    .rename(columns={"qtd_ocorrencias": "total_ocorrencias"})
)

print()
ranking

Mês de referência: 2026-03



,aisp,total_ocorrencias
0,18,67
1,25,59
2,14,44
3,28,43
4,8,42
5,41,41
6,15,40
7,27,39
8,24,37
9,20,34


## Célula 4 — Série temporal

Reutilizamos `analise_temporal()` sem filtros — agrega todos os crimes e
todas as AISPs em uma linha por mês, pronta para um gráfico de linha.

Não há sentido em reescrever essa lógica aqui: a função já foi testada
e validada no notebook 04.

In [4]:
series = analise_temporal(df)

print("Shape da série:", series.shape)
series.head(5)

2026-05-11 22:54:21 [INFO] Serie temporal: 279 meses | crime=todos | aisp=todas


Shape da série: (279, 3)


,total_ocorrencias,taxa_100k_media,media_movel_3
mes_ano,,,
2003-01,1929,1.570033,3.651455
2003-02,1904,1.555289,3.822421
2003-03,2010,1.648149,4.016534
2003-04,1944,1.614620,3.997047
2003-05,1965,1.580805,3.887139


## Célula 5 — Por tipo de crime

Agrega o total histórico por tipo de crime — útil para gráfico de barras
ou pizza que mostre a proporção de cada crime no período completo.

`reset_index()` converte o índice de volta em coluna, deixando o DataFrame
com formato padrão que o Plotly e o Streamlit esperam.

In [5]:
por_tipo = (
    df.groupby("tipo_crime")["qtd_ocorrencias"]
    .sum()
    .reset_index()
    .rename(columns={"qtd_ocorrencias": "total_ocorrencias"})
    .sort_values("total_ocorrencias", ascending=False)
)

por_tipo

,tipo_crime,total_ocorrencias
3,letalidade_violenta,137606
0,cvli,114260
1,hom_doloso,109621
2,latrocinio,3578


## Célula 6 — Montar e validar o dicionário

Juntamos os quatro DataFrames num dicionário. Isso é o que `preparar_visualizacao()`
vai retornar — e o que `main()` vai entregar para o dashboard.

Verificamos que todas as chaves existem e que nenhum DataFrame está vazio.

In [6]:
resultado = {
    "heatmap":  heatmap,
    "ranking":  ranking,
    "series":   series,
    "por_tipo": por_tipo,
}

for nome, dado in resultado.items():
    print(f"{nome:10s} → shape: {dado.shape}")

heatmap    → shape: (42, 279)
ranking    → shape: (10, 2)
series     → shape: (279, 3)
por_tipo   → shape: (4, 2)


---
## Por que a função não foi para o `pipeline.py`

A exploração neste notebook cumpriu seu papel: validou que as quatro
transformações funcionam e que os formatos resultantes são adequados
para Plotly e Streamlit.

A função não foi mantida porque o dashboard precisou de filtros dinâmicos.
Um DataFrame pré-computado sobre todos os dados não consegue responder
a "me mostre só AISP 5, crimes de roubo, entre 2018 e 2023" — o app
precisa refazer o agrupamento a cada interação do usuário.

O que ficou do trabalho deste notebook: os padrões de transformação
(`pivot_table`, `nlargest`, `groupby`) estão presentes no `app.py`,
aplicados sobre o `df` já filtrado.